In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DataCollatorWithPadding
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch

In [3]:
df = pd.read_csv("/kaggle/input/datasets/saurabhshahane/fake-news-classification/WELFake_Dataset.csv")
df["combined"] = df["title"].fillna("") + " " + df["text"].fillna("")
df["combined"] = df["combined"].str.strip()

print(df.duplicated(subset=["combined"]).sum())
print(df.duplicated(subset=["title"]).sum())

df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print("Original:", len(df))

df = df.drop_duplicates(subset=["combined", "label"]).reset_index(drop=True)

print("After dedup:", len(df))

8458
9786
Original: 72134
After dedup: 63676


In [4]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(len(train_df), len(val_df), len(test_df))

44573 9551 9552


In [5]:
train_set = set(train_df["combined"])

print("Train-Val overlap:",
      val_df["combined"].isin(train_set).sum())

print("Train-Test overlap:",
      test_df["combined"].isin(train_set).sum())

Train-Val overlap: 0
Train-Test overlap: 0


In [6]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["combined"],
        truncation=True,
        max_length=512
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_ds = Dataset.from_pandas(train_df[["combined", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["combined", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["combined", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/44573 [00:00<?, ? examples/s]

Map:   0%|          | 0/9551 [00:00<?, ? examples/s]

Map:   0%|          | 0/9552 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import math


batch_size = 16
epochs = 3
warmup_ratio = 0.1

steps_per_epoch = math.ceil(len(train_ds) / batch_size)
total_steps = steps_per_epoch * epochs
warmup_steps = int(total_steps * warmup_ratio)


args = TrainingArguments(
    output_dir="./roberta-fakenews",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,           
    logging_steps=100,
    report_to="none"
)

In [9]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.016585,0.047661,0.996650,0.996622
2,0.017807,0.014321,0.998953,0.998944
3,0.000029,0.010418,0.999372,0.999366


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4179, training_loss=0.06369938155779711, metrics={'train_runtime': 8149.9411, 'train_samples_per_second': 16.407, 'train_steps_per_second': 0.513, 'total_flos': 3.518294721168384e+16, 'train_loss': 0.06369938155779711, 'epoch': 3.0})

In [27]:
trainer.save_model("/kaggle/working/final_model")
tokenizer.save_pretrained("/kaggle/working/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/final_model/tokenizer_config.json',
 '/kaggle/working/final_model/tokenizer.json')

In [28]:
pred = trainer.predict(test_ds)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [29]:
import json

with open("test_metrics.json", "w") as f:
    json.dump(pred.metrics, f, indent=4)

In [30]:
y_pred = pred.predictions.argmax(axis=-1)
y_true = pred.label_ids

report = classification_report(y_true, y_pred)

with open("classification_report.txt", "w") as f:
    f.write(report)

In [31]:
cm = confusion_matrix(y_true, y_pred)

np.savetxt(
    "confusion_matrix.csv",
    cm,
    delimiter=",",
    fmt="%d"
)

In [32]:
results = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "logit_real": pred.predictions[:, 0],
    "logit_fake": pred.predictions[:, 1]
})

results.to_csv("predictions.csv", index=False)

In [33]:
import os

print(os.listdir("/kaggle/working"))

['predictions.csv', 'confusion_matrix.csv', 'classification_report.txt', 'final_model', 'roberta-fakenews', 'test_metrics.json', '.virtual_documents']


In [34]:
print(os.listdir("/kaggle/working/final_model"))

['tokenizer_config.json', 'training_args.bin', 'config.json', 'model.safetensors', 'tokenizer.json']


In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/final_model")
print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [36]:
import shutil

shutil.make_archive(
    "/kaggle/working/final_model",
    "zip",
    "/kaggle/working/final_model"
)

'/kaggle/working/final_model.zip'

In [37]:
import os

print(os.listdir("/kaggle/working"))

['predictions.csv', 'confusion_matrix.csv', 'classification_report.txt', 'final_model.zip', 'final_model', 'roberta-fakenews', 'test_metrics.json', '.virtual_documents']


In [45]:
import shutil

shutil.make_archive(
    "/kaggle/working/final_model",
    "zip",
    root_dir="/kaggle/working",
    base_dir="final_model"
)

'/kaggle/working/final_model.zip'

In [49]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("final_model")
tokenizer = AutoTokenizer.from_pretrained("final_model")

print("Loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded successfully!
